In [ ]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.model_selection import GroupShuffleSplit

from pathlib import Path

DATA_DIR = Path("../src/data")

DETAILS_PATH = DATA_DIR / "san_francisco_listings_details.csv"
TRAIN_S2_OUT = DATA_DIR / "train_s2_with_desc.csv"
TEST_S2_OUT  = DATA_DIR / "test_s2_with_desc.csv"

DETAILS_PATH, TRAIN_S2_OUT, TEST_S2_OUT


(WindowsPath('../src/data/san_francisco_listings_details.csv'),
 WindowsPath('../src/data/train_s2_with_desc.csv'),
 WindowsPath('../src/data/test_s2_with_desc.csv'))

In [3]:
df = pd.read_csv(DETAILS_PATH)

# Same filters as in 01_data_preprocessing
df = df.dropna(subset=['price'])
df = df[df['minimum_nights'] <= 30]

print("After price + min_nights filters:", df.shape)


After price + min_nights filters: (5374, 79)


In [4]:
keep_cols = [
    'id',  # <<< NEW compared to original
    'description',
    'neighbourhood_cleansed', 'latitude', 'longitude', 
    'room_type','accommodates', 'bathrooms', 'bedrooms', 'beds', 'amenities', 
    'price', 'number_of_reviews', 'review_scores_rating', 'host_is_superhost', 
    'availability_30', 'availability_365'
]

df = df[keep_cols]

print("Shape after column selection:", df.shape)
df.head()


Shape after column selection: (5374, 17)


,id,description,neighbourhood_cleansed,latitude,longitude,room_type,accommodates,bathrooms,bedrooms,beds,amenities,price,number_of_reviews,review_scores_rating,host_is_superhost,availability_30,availability_365
0,958,Our bright garden unit overlooks a lovely back...,Western Addition,37.77028,-122.43317,Entire home/apt,3,1.0,1.0,2.0,"[""Clothing storage: closet and dresser"", ""Esse...",$157.00,496,4.89,t,5,224
1,5858,We live in a large Victorian house on a quiet ...,Bernal Heights,37.74474,-122.42089,Entire home/apt,4,2.0,2.0,2.0,"[""Carbon monoxide alarm"", ""Shampoo"", ""Hair dry...",$250.00,105,4.87,f,22,357
2,8014,Room is on the second floor so it gets a good ...,Outer Mission,37.73077,-122.44827,Private room,1,2.0,1.0,1.0,"[""Dryer"", ""Essentials"", ""Hangers"", ""Backyard"",...",$67.00,90,4.77,t,0,42
4,8339,"For creative humans who love art, space, photo...",Western Addition,37.77377,-122.43614,Entire home/apt,2,1.5,1.0,1.0,"[""Dedicated workspace"", ""Essentials"", ""Hangers...",$527.00,25,4.86,f,1,327
5,10537,Casa de Paz (House of Peace) is like staying w...,Bayview,37.71750,-122.39698,Private room,2,1.5,1.0,1.0,"[""Blender"", ""Shared BBQ grill: charcoal"", ""Ess...",$108.00,40,4.97,t,30,363


In [5]:
# Price cleaning
df['price'] = df['price'].replace('[\$,]', '', regex=True).astype(float)
df['price'] = df['price'].round(0).astype(int)
df = df[df['price'] > 0]

# Rating-related features
df['no_rating'] = (df['number_of_reviews'] == 0).astype(int)
df['review_scores_rating'].fillna(0, inplace=True)
df = df.dropna(subset=['bathrooms', 'bedrooms', 'beds', 'host_is_superhost'])

# Amenities
df['amenity_count'] = df['amenities'].apply(lambda x: len(str(x).split(',')))
df.drop(columns=['amenities'], inplace=True)

# Rename neighbourhood column for consistency
df.rename(columns={'neighbourhood_cleansed': 'neighbourhood'}, inplace=True)

print("After preprocessing:", df.shape)


After preprocessing: (5209, 18)


C:\Users\ethan\AppData\Local\Temp\ipykernel_24272\852910174.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['review_scores_rating'].fillna(0, inplace=True)


In [6]:
df['description_clean'] = (
    df['description']
    .fillna('')
    .str.replace(r'<br\s*/?>', ' ', regex=True)
    .str.replace(r'[^A-Za-z0-9\s]', '', regex=True)
    .str.lower()
    .str.strip()
)
df['desc_word_count'] = df['description_clean'].apply(lambda x: len(x.split()))


In [7]:
# Cap extreme prices at 99.5th percentile
X = np.percentile(df['price'], 99.5)
df = df[df['price'] <= X]

# Log transform
df['log_price'] = np.log(df['price'])
df.drop(columns=['price'], inplace=True)

print("After capping & log transform:", df.shape)

After capping & log transform: (5182, 20)


In [8]:
# Avoid 0
df['accommodates'] = df['accommodates'].clip(lower=1)

df['bedrooms_per_guest']  = df['bedrooms']   / df['accommodates']
df['bathrooms_per_guest'] = df['bathrooms']  / df['accommodates']
df['beds_per_guest']      = df['beds']       / df['accommodates']

# Drop collinear raw counts
df.drop(columns=['bedrooms', 'bathrooms', 'beds'], inplace=True)

# Drop redundant no_rating
df.drop(columns=['no_rating'], inplace=True)

In [9]:
lp_min = df['log_price'].min()
lp_max = df['log_price'].max()
df['log_price_norm'] = (df['log_price'] - lp_min) / (lp_max - lp_min + 1e-12)


In [10]:
coords = df[['latitude', 'longitude']]

kmeans = KMeans(n_clusters=25, random_state=42, n_init='auto')
df['geo_cluster'] = kmeans.fit_predict(coords)

df['geo_cluster'].value_counts().head()

c:\Users\ethan\OneDrive\Bureau\Stanford\Courses\CS229\Project\.conda\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


geo_cluster
4     681
10    449
18    293
16    286
6     261
Name: count, dtype: int64

In [11]:
from sklearn.model_selection import GroupShuffleSplit

X = df.drop(columns=['log_price', 'description', 'description_clean', 'log_price_norm', 'desc_word_count'])
y = df['log_price']
groups = df['geo_cluster']

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

print('Train clusters:', len(set(groups.iloc[train_idx])))
print('Test clusters :', len(set(groups.iloc[test_idx])))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


Train clusters: 20
Test clusters : 5


In [12]:
# Use the *full* df rows for each split, not just X
train_s2_with_desc = df.iloc[train_idx].copy()
test_s2_with_desc  = df.iloc[test_idx].copy()

# For convenience, ensure log_price is present (it already is)
assert 'log_price' in train_s2_with_desc.columns

print("Train_s2_with_desc:", train_s2_with_desc.shape)
print("Test_s2_with_desc :", test_s2_with_desc.shape)

train_s2_with_desc.to_csv(TRAIN_S2_OUT, index=False)
test_s2_with_desc.to_csv(TEST_S2_OUT, index=False)

TRAIN_S2_OUT, TEST_S2_OUT


Train_s2_with_desc: (4402, 21)
Test_s2_with_desc : (780, 21)


(WindowsPath('../src/data/train_s2_with_desc.csv'),
 WindowsPath('../src/data/test_s2_with_desc.csv'))

In [13]:
TRAIN_PATH = DATA_DIR / "train_s2_with_desc.csv"
TEST_PATH  = DATA_DIR / "test_s2_with_desc.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

In [26]:
import sys
import os
sys.path.append(os.path.abspath("../"))

from src.features.build_description import build_description_embeddings

train_text = train_df["description"].fillna("").astype(str)
test_text  = test_df["description"].fillna("").astype(str)

emb_train, emb_test, sbert_model = build_description_embeddings(
    train_text,
    test_text
)

OSError: [WinError 127] La procédure spécifiée est introuvable. Error loading "c:\Users\ethan\OneDrive\Bureau\Stanford\Courses\CS229\Project\.conda\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [ ]:
out_dir = os.path.abspath(os.path.join("..", "src", "data", "processed"))
os.makedirs(out_dir, exist_ok=True)

np.save(os.path.join(out_dir, "train_desc_embeddings.npy"), emb_train)
np.save(os.path.join(out_dir, "test_desc_embeddings.npy"), emb_test)